# Dimensionality Reduction


## Imports


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os


## Inputs


In [ ]:
# --- Load descriptor matrix and metadata
path_to_results = Path("desc")

npy_file = max(path_to_results.glob("*.npy"), key=lambda p: p.stat().st_mtime)
all_descriptors = np.load(npy_file)
print(f"Loaded descriptors from {npy_file.name}")
print(f"Descriptors shape: {all_descriptors.shape}")

# --- Load provenance table (parquet preferred, CSV fallback)
provenance_files = list(path_to_results.glob("*.parquet")) + list(path_to_results.glob("*.csv"))
if not provenance_files:
    raise FileNotFoundError("No provenance table found in desc/")
provenance_path = next((p for p in provenance_files if p.suffix == ".parquet"), provenance_files[0])
if provenance_path.suffix == ".parquet":
    metadata_df = pd.read_parquet(provenance_path)
else:
    metadata_df = pd.read_csv(provenance_path)
print(f"Loaded provenance metadata from {provenance_path.name}")

# --- Load filemap
json_files = [p for p in path_to_results.glob("*.json") if "filemap" in p.name]
if not json_files:
    raise FileNotFoundError("No filemap JSON found in desc/")
filemap_path = json_files[0]
with filemap_path.open("r") as f:
    filemap = json.load(f)
print(f"Loaded filemap from {filemap_path.name}")

# --- Load run config (optional) to detect appended forces
config_files = list(path_to_results.glob("*_config.json"))
config = {}
if config_files:
    config_path = max(config_files, key=lambda p: p.stat().st_mtime)
    with config_path.open("r") as f:
        config = json.load(f)
    print(f"Loaded run config from {config_path.name}")

soap_dim = config.get("soap_dim")
force_dim = config.get("force_dim", 0)
include_forces = config.get("include_forces", False)

# Choose whether to include appended force components in dimensionality reduction
use_forces_in_dimred = False  # set False to use only SOAP part
if include_forces and soap_dim and force_dim:
    print(f"Descriptor split: SOAP={soap_dim}, forces={force_dim}")
    if use_forces_in_dimred:
        print("Using SOAP+forces for dimensionality reduction")
    else:
        all_descriptors = all_descriptors[:, :soap_dim]
        print(f"Using SOAP-only descriptors: {all_descriptors.shape}")


## Standardize descriptors


In [ ]:
# --- Standardize the descriptors
scaler = StandardScaler()
standardized_descriptors = scaler.fit_transform(all_descriptors)
print("Descriptors standardized.")
print(f"Standardized descriptors shape: {standardized_descriptors.shape}")

## Reduce dimensions and save outputs


In [ ]:
X = standardized_descriptors

# Choose: "pca", "umap", or "tsne"
method = "pca"          # change as needed
random_state = 42

# Optional: fast pre-reduction for UMAP/t-SNE on large SOAP
pca_prereduce_dim = None   # set None to skip

embedding = None
model = None

os.makedirs("embedding", exist_ok=True)

if method.lower() == "pca":
    # make a directory named embedding to save outputs
    
    n_components = 0.95         # embedding dimension
    model = PCA(n_components=n_components, random_state=random_state)
    embedding = model.fit_transform(X)
    save_pca = True
    if save_pca:
        with open("embedding/pca_model.json", "w") as f:
            json.dump({
                "components": model.components_.tolist(),
                "explained_variance": getattr(model, "explained_variance_", []).tolist() if hasattr(model, "explained_variance_") else [],
                "explained_variance_ratio": getattr(model, "explained_variance_ratio_", []).tolist() if hasattr(model, "explained_variance_ratio_") else [],
                "mean": getattr(model, "mean_", []).tolist() if hasattr(model, "mean_") else [],
                "n_components": int(getattr(model, "n_components_", getattr(model, "n_components", 0))),
                "n_features": int(getattr(model, "n_features_in_", X.shape[1])),
            }, f, indent=2) 
        print("Saved PCA model to embedding/pca_model.json")
    save_pca_matrix = True
    if save_pca_matrix:
        np.save("embedding/pca_embedding.npy", embedding)
        print("Saved PCA embedding matrix to embedding/pca_embedding.npy")

elif method.lower() == "umap":
    n_components = 10
    try:
        import umap.umap_ as umap
    except ImportError:
        raise RuntimeError("UMAP not installed. pip install umap-learn")

    X_in = X
    if pca_prereduce_dim:
        X_in = PCA(n_components=min(pca_prereduce_dim, X.shape[1]), random_state=random_state).fit_transform(X)

    model = umap.UMAP(
        n_components=n_components,
        n_neighbors=15,        # tune per dataset size
        min_dist=0.0,
        metric="euclidean",
        random_state=random_state,
        verbose=True
    )
    embedding = model.fit_transform(X_in)

elif method.lower() == "tsne":
    n_components = 10 
    from sklearn.manifold import TSNE
    # t-SNE is O(N^2). Use PCA pre-step by default.
    X_in = X
    if pca_prereduce_dim:
        X_in = PCA(n_components=min(pca_prereduce_dim, X.shape[1]), random_state=random_state).fit_transform(X)

    model = TSNE(
        n_components=n_components,
        perplexity=30,         # 5–50 typical
        n_iter=1000,
        learning_rate="auto",
        init="pca",
        random_state=random_state,
        verbose=1,
        method="barnes_hut" if X_in.shape[0] < 50000 else "exact"
    )
    embedding = model.fit_transform(X_in)

else:
    raise ValueError("method must be 'pca', 'umap', or 'tsne'")

print(f"{method.upper()} embedding shape:", embedding.shape)

## Optional: visualize embeddings


In [ ]:
# color by species only
if "symbol" not in metadata_df.columns:
    raise KeyError("Provenance parquet must contain 'symbol' column.")

labels = metadata_df["symbol"].astype(str).astype("category")
cats = list(labels.cat.categories)
codes = labels.cat.codes.to_numpy()

plt.figure(figsize=(8, 6))
if embedding.shape[1] >= 2:
    for lab in cats:
        idx = (labels == lab)
        if not np.any(idx):
            continue
        plt.scatter(embedding[idx, 0], embedding[idx, 1],
                    s=1, alpha=0.1, label=lab)
    plt.xlabel("Component 1", fontsize=16)
    plt.ylabel("Component 2", fontsize=16)
    plt.title(f"{method.upper()} Embedding of SOAP Descriptors", fontsize=16)
    plt.grid(True, linewidth=0.2)
    plt.legend(markerscale=3, frameon=False, loc="best", title="symbol", fontsize=14)
    plt.show()


## Optional: inspect structures in a region


In [ ]:
from src.dim_red_utils import select_by_pca_box, origins_from_mask, group_summary

# Example usage
box = {0: (20, 25), 1: (-35, -25)}
# box is your PCA region, e.g. {0:(-5,-2), 1:(1,3)}
mask = select_by_pca_box(embedding, box)

# Resolve provenance for selected points
sel = origins_from_mask(mask, metadata_df, filemap)

# Keep exactly what you need (+ optional index in embedding)
sel_min = sel.loc[:, ["file_path", "struct_id", "atom_id"]].copy()
sel_min["row_index"] = np.nonzero(mask)[0]  # optional: row position in embedding/descriptors

print(f"Selected atoms: {len(sel_min)}")
print(sel_min.head())
